In [ ]:
!pip install autogluon -q

In [5]:
import numpy

In [6]:
from sklearn import datasets 

In [7]:
datasets.load_iris

<function sklearn.datasets._base.load_iris(*, return_X_y=False, as_frame=False)>

In [5]:
from sklearn.datasets import load_iris
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

# Load iris dataset from sklearn
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

# Combine features and target into one DataFrame
df = pd.concat([X, y], axis=1)

# Split into train and test
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)


In [6]:
%%time
# Train AutoGluon TabularPredictor
predictor = TabularPredictor(label="target", problem_type="multiclass").fit(train_data)


No path specified. Models will be saved in: "AutogluonModels\ag-20250630_104450"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          12
Memory Avail:       4.63 GB / 15.33 GB (30.2%)
Disk Space Avail:   326.38 GB / 475.94 GB (68.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'       

CPU times: total: 1min 5s
Wall time: 29 s


In [7]:
# Make predictions on the test set
predictions = predictor.predict(test_data)

# Evaluate accuracy
accuracy = (predictions == test_data["target"]).mean()
print(f"Test Accuracy: {accuracy:.4f}")


Test Accuracy: 1.0000


In [8]:
# Show all models and their performance
predictor.leaderboard(test_data, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetFastAI,1.000000,1.000000,accuracy,0.018887,0.013303,5.123322,0.018887,0.013303,5.123322,1,True,3
1,WeightedEnsemble_L2,1.000000,1.000000,accuracy,0.022929,0.013303,5.288027,0.004043,0.000000,0.164704,2,True,14
2,LightGBMXT,1.000000,1.000000,accuracy,0.023446,0.010297,0.720730,0.023446,0.010297,0.720730,1,True,4
3,CatBoost,1.000000,1.000000,accuracy,0.024976,0.001670,0.932276,0.024976,0.001670,0.932276,1,True,8
4,KNeighborsDist,1.000000,0.958333,accuracy,0.045241,0.032916,0.012141,0.045241,0.032916,0.012141,1,True,2
5,LightGBMLarge,1.000000,1.000000,accuracy,0.057590,0.014047,1.015258,0.057590,0.014047,1.015258,1,True,13
6,NeuralNetTorch,1.000000,1.000000,accuracy,0.065332,0.000000,6.257907,0.065332,0.000000,6.257907,1,True,12
7,ExtraTreesGini,1.000000,0.958333,accuracy,0.126285,0.097236,1.346769,0.126285,0.097236,1.346769,1,True,9
8,ExtraTreesEntr,1.000000,0.958333,accuracy,0.127344,0.104283,1.369891,0.127344,0.104283,1.369891,1,True,10
9,RandomForestEntr,1.000000,0.958333,accuracy,0.129566,0.105077,1.352091,0.129566,0.105077,1.352091,1,True,7


In [9]:
lb = predictor.leaderboard(silent=True)
best_model = lb.loc[0, 'model']
print("Best model:", best_model)



Best model: NeuralNetTorch


In [10]:
# Show leaderboard sorted by performance (higher is better)
lb = predictor.leaderboard(silent=True)

# Display the best model
best_model = lb.loc[0, 'model']
print("Best model:", best_model)


Best model: NeuralNetTorch


In [14]:
model_object = predictor._trainer.load_model(best_model)
print(model_object)


In [17]:
print(model_object.params)


{'num_epochs': 1000, 'epochs_wo_improve': None, 'activation': 'relu', 'embedding_size_factor': 1.0, 'embed_exponent': 0.56, 'max_embedding_dim': 100, 'y_range': None, 'y_range_extend': 0.05, 'dropout_prob': 0.1, 'optimizer': 'adam', 'learning_rate': 0.0003, 'weight_decay': 1e-06, 'proc.embed_min_categories': 4, 'proc.impute_strategy': 'median', 'proc.max_category_levels': 100, 'proc.skew_threshold': 0.99, 'use_ngram_features': False, 'num_layers': 4, 'hidden_size': 128, 'max_batch_size': 512, 'use_batchnorm': False, 'loss_function': 'auto'}


In [31]:
from sklearn.metrics import accuracy_score

y_true = test_data['target']
y_pred = predictor.predict(test_data, model=best_model)

print("Accuracy:", accuracy_score(y_true, y_pred))


Accuracy: 1.0


In [32]:
submission = test_data.copy()  # or whatever ID column
submission['Prediction'] = preds
submission.to_csv('submission.csv', index=False)

NameError: name 'preds' is not defined

In [ ]:
# If no ID column exists:
test_data = test_data.copy()
test_data['id'] = range(len(test_data))

# Now create submission:
submission = test_data[['id']].copy()
submission['Prediction'] = preds

# Save to CSV:
submission.to_csv('submission.csv', index=False)


NameError: name 'preds' is not defined